# Nortek Aquadopp Data Processing

Processing path used: 

proc_1 IMOS NetCDF -> QC flagging -> proc_2 IMOS NetCDF

### Setup

Imports

In [ ]:
import os
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

Import local tools

In [ ]:
TOOLS_DIR = Path.cwd().resolve().parent
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

# For IMOS NetCDF conversion
from tools.imos_nc_converter import imos_converter as imos_converter_module
importlib.reload(imos_converter_module)
IMOSNetCDFConverter_AQD = imos_converter_module.IMOSNetCDFConverter_AQD

# Read the metadata table
from tools import database_lookup as database_lookup_module
importlib.reload(database_lookup_module)
get_instrument_context = database_lookup_module.get_instrument_context
update_metadata_file_fields = database_lookup_module.update_metadata_file_fields

# Generic helpers
from tools.helpers import plot_data_by_qc

# QC flagging
from tools.helpers import apply_qc_flag_windows, write_manual_qc_flags_txt

Definitions

In [ ]:
# Working directory
os.chdir("/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/ash")

In [ ]:
# Select instrument using ID from satellite_altimetry_moorings_metadata.csv

inst_deploy_id = 258    # rec_202502/BASS3A_PTSUV

database, _row, cfg, metadata = get_instrument_context(
    inst_deploy_id=inst_deploy_id,
    # metadata_csv="/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/reference_mooring_proc_info/satellite_altimetry_moorings_metadata.csv",
    print_details=True,
)

In [ ]:
converter = IMOSNetCDFConverter_AQD(input_folder="", input_file="", output_dir="")

### QC

Read proc_1 dataset

In [ ]:
def resolve_stage_dir(path_value):
    stage_dir = Path(str(path_value)).expanduser()
    if not stage_dir.is_absolute():
        stage_dir = (Path.cwd() / stage_dir).resolve()
    return stage_dir

def find_proc_1_file(stage_dir):
    configured_name = _row.get("proc_1_file", None)
    if pd.notna(configured_name) and str(configured_name).strip():
        configured_name = str(configured_name).strip()
        configured_path = stage_dir / configured_name
        if configured_path.exists():
            return configured_path

    proc_1_pattern = f"{cfg['location']}_*_AQD_{int(_row['inst_id'])}_*.nc"
    proc_1_candidates = sorted(stage_dir.glob(proc_1_pattern))
    if not proc_1_candidates:
        proc_1_candidates = sorted(stage_dir.glob("*.nc"))
    if not proc_1_candidates:
        raise FileNotFoundError(f"No proc_1 NetCDF files found in {stage_dir}")
    return proc_1_candidates[-1]

proc_1_dir = resolve_stage_dir(_row["proc_1_path"])
proc_1_path = find_proc_1_file(proc_1_dir)
with xr.open_dataset(proc_1_path) as opened_ds:
    ds_proc1 = opened_ds.load()

# Ensure TIME is UTC-aware
if ds_proc1['TIME'].dt.tz is None:
    ds_proc1['TIME'] = ds_proc1['TIME'].dt.tz_localize('UTC')
else:
    ds_proc1['TIME'] = ds_proc1['TIME'].dt.tz_convert('UTC')

print(f"Using proc_1 input: {proc_1_path}")

In [ ]:
time_coverage_start = _row.get("time_coverage_start", None)
time_coverage_end = _row.get("time_coverage_end", None)
deploy_start = _row.get("deploy_date", None)
deploy_end = _row.get("recovery_date", None)


if "NOMINAL_DEPTH" in ds_proc1:
    converter_depth = float(np.asarray(ds_proc1["NOMINAL_DEPTH"].values).squeeze())
else:
    converter_depth = float(_row.get("nominal_depth", 0.0))

print(f"deploy_start: {deploy_start}")
print(f"time_coverage_start: {time_coverage_start}")
print(f"time_coverage_end: {time_coverage_end}")
print(f"deploy_end: {deploy_end}")

file_variables = list(ds_proc1.variables)
print("file variables:", file_variables)

### Proc_2

Manual QC - flagging

In [ ]:
ds_proc2 = ds_proc1.copy()

In [ ]:
PLOT_VARS = None  # set None for defaults, to specify use ["UCUR", "VCUR", "TEMP", "DEPTH", "CNDC", "PSAL", "PRES"] 
fig = plot_data_by_qc(
    ds_proc2,
    variables=PLOT_VARS,
    # flags_to_plot=[1],      # optional
    # y_zoom_to_good=True,    # optional
)
fig.show()

In [ ]:
manual_qc_flags = [
    # {"qc_vars": ["_quality_control"], "flag": 3, "start": "yyyy-mm-dd hh:mm:ss", "end": "yyyy-mm-dd hh:mm:ss", "comment": "",},
    {"qc_vars": ["DEPTH_quality_control"], "flag": 4, "start": "2024-07-31 06:00:00", "end": "2025-08-23 04:00:00", "comment": "sensor failed",},
]

ds_proc2 = apply_qc_flag_windows(ds_proc2, manual_qc_flags, time_name="TIME")


Save QC to proc_2 dataset and output

In [ ]:
# For proc_2, you're working with an ALREADY-CREATED IMOS NetCDF from proc_1
# So you need to use the converter in "publish" mode (copy + rename)
# But first, you need to save your QC-modified dataset
# # Trim to deployment window before creating NetCDF
ds_proc2_trimmed = ds_proc2.sel(TIME=slice(time_coverage_start, time_coverage_end))

In [ ]:
# Convert TIME from datetime64 to numeric (days since 1950-01-01)
time_numeric = (ds_proc2_trimmed['TIME'].values - np.datetime64('1950-01-01')) / np.timedelta64(1, 'D')

In [ ]:
proc_2_out = converter.process(
    time_data=time_numeric,
    temp_data=ds_proc2_trimmed['TEMP'].values,
    temp_qc_data=ds_proc2_trimmed['TEMP_quality_control'].values,
    ucur_data=ds_proc2_trimmed['UCUR'].values,
    ucur_qc_data=ds_proc2_trimmed['UCUR_quality_control'].values,
    vcur_data=ds_proc2_trimmed['VCUR'].values,
    vcur_qc_data=ds_proc2_trimmed['VCUR_quality_control'].values,
    depth_data=ds_proc2_trimmed['DEPTH'].values if 'DEPTH' in ds_proc2_trimmed else None,
    depth_qc_data=ds_proc2_trimmed['DEPTH_quality_control'].values if 'DEPTH_quality_control' in ds_proc2_trimmed else None,
    longitude=float(cfg["longitude"]),
    latitude=float(cfg["latitude"]),
    depth=converter_depth,
    inst_channels=str(cfg.get("inst_channels", "")),
    start_of_good_data=time_coverage_start,
    site_code=str(cfg["location"]),
    version=str(cfg.get("version", "1")),
    instrument=str(cfg.get("inst_type", "")),
    inst_id=str(int(cfg["inst_id"])),
    location=cfg["location"],
    output_name_mode="internal",
    output_stage="proc_2",
    metadata_row=_row,
    metadata_mode="fill_missing",
    deployment_id=cfg.get("deployment_id", ""),
    mooring_channels=cfg.get("mooring_channels", ""),
    nominal_inst_depth=cfg.get("nominal_inst_depth", ""),
    deploy_date=cfg.get("deploy_date", ""),
    recovery_date=cfg.get("recovery_date", ""),
    time_coverage_start=time_coverage_start,
    time_coverage_end=time_coverage_end,
)

manual_qc_log_path = write_manual_qc_flags_txt(proc_2_out, manual_qc_flags)

print(f"proc_2 output: {proc_2_out}")
print(f"manual QC flags log: {manual_qc_log_path}")

Write proc_2 filename to metadata table

In [ ]:
proc_2_name = Path(proc_2_out).name
_row = update_metadata_file_fields(inst_deploy_id, {"proc_2_file": proc_2_name}, output_paths={"proc_2_file": proc_2_out}, working_dir=Path.cwd())
print(f"Updated proc_2_file: {_row['proc_2_file']}")